In [1]:
from dotenv import load_dotenv
import os
from langchain_groq import ChatGroq
from google import genai
from google.genai import types
import time

load_dotenv()
groq_key = os.getenv("GROQ_API_KEY")
tavily_key = os.getenv("TAVILY_API_KEY")
gemini_key = os.getenv("GEMINI_API_KEY")

llm = ChatGroq(model="qwen/qwen3.8-27b", temperature=0)
gemini_client = genai.Client(api_key=gemini_key)

print(" Setup complete. Tracing:", os.getenv("LANGSMITH_TRACING"))

 Setup complete. Tracing: true


In [4]:
from tavily import TavilyClient
tavily_client = TavilyClient(api_key=tavily_key)

def research_with_fallback(query: str) -> dict:
    """Tries Gemini (grounded search + citations) first. Falls back to Tavily
    automatically if Gemini is unavailable, so the pipeline never fully blocks
    on one provider's outage."""
    try:
        resp = gemini_client.models.generate_content(
            model="gemini-3.6-flash",
            contents=query,
            config=types.GenerateContentConfig(tools=[types.Tool(google_search=types.GoogleSearch())])
        )
        sources = []
        if resp.candidates[0].grounding_metadata:
            sources = [{"title": c.web.title, "url": c.web.uri} for c in resp.candidates[0].grounding_metadata.grounding_chunks]
        return {"answer": resp.text, "sources": sources, "provider": "gemini"}
    except Exception as e:
        print(f" Gemini unavailable ({e.__class__.__name__}), falling back to Tavily...")
        results = tavily_client.search(query=query, max_results=3)
        sources = [{"title": r["title"], "url": r["url"]} for r in results["results"]]
        answer = " ".join(r.get("content", "")[:300] for r in results["results"])
        return {"answer": answer, "sources": sources, "provider": "tavily"}

result = research_with_fallback("latest solid-state EV battery safety developments 2026")
print("Provider used:", result["provider"])
print("Answer:", result["answer"][:200])
print("Sources:", result["sources"][:2])

 Gemini unavailable (ClientError), falling back to Tavily...
Provider used: tavily
Answer: With advances in EV battery technology, users will experience improved performance from both battery types in 2026. Another key use case is enhanced safety and enabling future EV range and charging in
Sources: [{'title': 'Solid-State vs Semi-Solid EV Batteries: 2026 Reality Check', 'url': 'https://www.polarismarketresearch.com/blog/solid-state-vs-semi-solid-ev-batteries-who-is-actually-shipping'}, {'title': 'Solid-state battery technology', 'url': 'https://www.evinfrastructurenews.com/ev-battery/solid-state-battery-technology'}]


In [3]:
try:
    resp = gemini_client.models.generate_content(
        model="gemini-3.6-flash",
        contents="test"
    )
    print(resp.text)
except Exception as e:
    print(type(e).__name__, ":", e)

Test received! Everything seems to be working properly. 

How can I help you today?


In [5]:
def research_with_fallback(query: str) -> dict:
    try:
        resp = gemini_client.models.generate_content(
            model="gemini-3.6-flash",
            contents=query,
            config=types.GenerateContentConfig(tools=[types.Tool(google_search=types.GoogleSearch())])
        )
        sources = []
        if resp.candidates[0].grounding_metadata:
            sources = [{"title": c.web.title, "url": c.web.uri} for c in resp.candidates[0].grounding_metadata.grounding_chunks]
        return {"answer": resp.text, "sources": sources, "provider": "gemini"}
    except Exception as e:
        print(f" Gemini unavailable — {type(e).__name__}: {e}")   # ← full detail, not just the type
        results = tavily_client.search(query=query, max_results=3)
        sources = [{"title": r["title"], "url": r["url"]} for r in results["results"]]
        answer = " ".join(r.get("content", "")[:300] for r in results["results"])
        return {"answer": answer, "sources": sources, "provider": "tavily"}

result = research_with_fallback("latest solid-state EV battery safety developments 2026")
print("\nProvider used:", result["provider"])

 Gemini unavailable — ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. ', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}]}}

Provider used: tavily


In [6]:
"""
SUPERVISOR PATTERN
One central node sees everything and decides which specialist to call next.
Specialists don't talk to each other — every decision flows through the supervisor.

    User → Supervisor → Agent A
                      → Agent B
                      → Agent C
           (supervisor decides each hop, one at a time)

Pros: easy to reason about, easy to debug (one place decides everything),
      easy to add new agents (just teach the supervisor about them)
Cons: supervisor can become a bottleneck; every decision costs an extra LLM call
"""

from typing import TypedDict, Literal

class SupervisorState(TypedDict):
    query: str
    next_agent: str
    results: list

def supervisor_node(state: SupervisorState) -> dict:
    prompt = f"""Given this state, which agent should run next: 'researcher', 'fact_checker',
'summarizer', or 'done'? Query: {state['query']}. Results so far: {len(state['results'])}
Reply with just one word."""
    decision = llm.invoke(prompt).content.strip().lower()
    print(f" Supervisor decided: {decision}")
    return {"next_agent": decision}

print(" Supervisor pattern skeleton defined")

 Supervisor pattern skeleton defined


In [7]:
"""
HIERARCHICAL PATTERN
A top-level supervisor delegates to SUB-supervisors, each managing their own
team of specialists. Useful when you have distinct "departments."

    User → Top Supervisor → Research Team Lead → Web Search Agent
                                                → Data Extraction Agent
                          → Writing Team Lead   → Summarizer Agent
                                                → Report Generator Agent

Pros: scales better for large agent counts, mirrors real org structures,
      each sub-supervisor only needs to know its own team
Cons: more moving parts, deeper call chains mean more latency per decision,
      harder to debug since a decision might span 2+ levels
"""

def research_team_lead(state: dict) -> dict:
    print(" Research Team Lead deciding between web_search and data_extraction...")
    return {"delegated_to": "web_search_agent"}

def writing_team_lead(state: dict) -> dict:
    print(" Writing Team Lead deciding between summarizer and report_generator...")
    return {"delegated_to": "summarizer_agent"}

print(" Hierarchical pattern skeleton defined")

 Hierarchical pattern skeleton defined


In [8]:
"""
NETWORK PATTERN
No central authority. Any agent can hand off directly to any other agent
based on its own judgment of what's needed next.

    Web Search Agent ⇄ Fact-Checker ⇄ Summarizer
              ⇅                            ⇅
        Data Extraction  ⇄⇄⇄⇄⇄⇄⇄⇄⇄  Report Generator

Pros: no bottleneck, very flexible, agents can loop back to each other freely
Cons: much harder to debug ("who called whom, and why?"), easy to create
      infinite loops or unpredictable paths, hard to guarantee the pipeline
      terminates in a reasonable number of steps
"""

def web_search_agent_net(state: dict) -> dict:
    
    print(" Web Search Agent deciding its own next hop...")
    return {"handoff_to": "fact_checker"}

print(" Network pattern skeleton defined (illustrative only — not building this one)")

 Network pattern skeleton defined (illustrative only — not building this one)


In [9]:
decision = """
ARCHITECTURE DECISION — Day 5

Chosen pattern: SUPERVISOR

Reasoning:
1. This project has a FIXED, known set of 5 specialist agents — not an open-ended
   or growing team — so Hierarchical's extra structure (team leads) adds
   complexity without a real benefit here.
2. Debuggability matters a lot for a learning project and for a portfolio piece
   a mentor/interviewer will review — Supervisor keeps every routing decision
   in ONE place, visible in one LangSmith trace, not scattered across peer
   handoffs like Network would produce.
3. The research process has a natural, mostly-linear flow anyway
   (plan → research → fact-check → summarize → report), which Supervisor
   handles cleanly via conditional edges — we already built exactly this
   pattern back on Day 2.
4. Network's flexibility is real, but its main risk (infinite loops, unclear
   termination) is exactly the kind of bug that's hard to fix under a
   30-day deadline.

Not fully ruling out Hierarchical later: if the Fact-Checking agent grows complex
enough to need its own sub-agents (e.g. one for cross-referencing, one for
source-credibility scoring), a small hierarchical pocket INSIDE the supervisor
pattern is a reasonable future extension — not a full rewrite.
"""
print(decision)


ARCHITECTURE DECISION — Day 5

Chosen pattern: SUPERVISOR

Reasoning:
1. This project has a FIXED, known set of 5 specialist agents — not an open-ended
   or growing team — so Hierarchical's extra structure (team leads) adds
   complexity without a real benefit here.
2. Debuggability matters a lot for a learning project and for a portfolio piece
   a mentor/interviewer will review — Supervisor keeps every routing decision
   in ONE place, visible in one LangSmith trace, not scattered across peer
   handoffs like Network would produce.
3. The research process has a natural, mostly-linear flow anyway
   (plan → research → fact-check → summarize → report), which Supervisor
   handles cleanly via conditional edges — we already built exactly this
   pattern back on Day 2.
4. Network's flexibility is real, but its main risk (infinite loops, unclear
   termination) is exactly the kind of bug that's hard to fix under a
   30-day deadline.

Not fully ruling out Hierarchical later: if the Fact-

In [ ]:
mermaid_diagram = """
graph TD
    A[User Query] --> B[Supervisor/Planner Agent — Groq]
    B -->|needs research| C[Web Search Agent — Gemini, Tavily fallback]
    B -->|needs extraction| D[Data Extraction Agent — Groq]
    B -->|needs verification| E[Fact-Checking Agent — Groq]
    B -->|needs summary| F[Summarization Agent — Groq]
    B -->|ready to finalize| G[Report Generator Agent — Groq]
    C --> B
    D --> B
    E --> B
    F --> B
    G --> H[Final Report]
"""
print(mermaid_diagram)

import os
os.makedirs("docs", exist_ok=True)

with open("docs/architecture.md", "w") as f:
    f.write("# System Architecture\n\n```mermaid\n" + mermaid_diagram + "\n```\n")
print("✅ Saved to docs/architecture.md")


graph TD
    A[User Query] --> B[Supervisor/Planner Agent]
    B -->|needs research| C[Web Search Agent — Gemini w/ Tavily fallback]
    B -->|needs extraction| D[Data Extraction Agent]
    B -->|needs verification| E[Fact-Checking Agent]
    B -->|needs summary| F[Summarization Agent]
    B -->|ready to finalize| G[Report Generator Agent]
    C --> B
    D --> B
    E --> B
    F --> B
    G --> H[Final Report]

 Saved to docs/architecture.md
